<a href="https://colab.research.google.com/github/Aakash-77089/Computer-Vision/blob/main/Global%20Average%20Pooling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Load and Preprocess Data (CIFAR-10)

We'll use the CIFAR-10 dataset, which consists of 60,000 32x32 color images in 10 classes, with 6,000 images per class. There are 50,000 training images and 10,000 test images.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import numpy as np
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

# Get class names for visualization (optional)
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Train images shape: {train_images.shape}")
print(f"Train labels shape: {train_labels.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Test labels shape: {test_labels.shape}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Train images shape: (50000, 32, 32, 3)
Train labels shape: (50000, 1)
Test images shape: (10000, 32, 32, 3)
Test labels shape: (10000, 1)


### 2. Build the CNN Model with Global Average Pooling (GAP)

Here, we define a CNN architecture. Notice how `GlobalAveragePooling2D` replaces the traditional `Flatten` layer and subsequent Dense layers before the final output layer. This significantly reduces the number of parameters and can help prevent overfitting.

In [ ]:
# Define the CNN model using Sequential API
model_gap = models.Sequential([
    # Input layer
    layers.Input(shape=(32, 32, 3)),

    # Convolutional Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Convolutional Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Convolutional Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Global Average Pooling Layer
    # This layer takes the average of each feature map.
    # For a feature map of size HxWxD, it outputs a vector of size D.
    layers.GlobalAveragePooling2D(),

    # Output layer
    # Since CIFAR-10 has 10 classes, the output layer has 10 units
    # with a softmax activation for multi-class classification.
    layers.Dense(10, activation='softmax')
])

# Display the model summary to see the architecture and parameter count
model_gap.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 290,090 (1.11 MB)

 Trainable params: 289,194 (1.10 MB)

 Non-trainable params: 896 (3.50 KB)

### 3. Compile the Model

We compile the model by specifying the optimizer, loss function, and metrics. For multi-class classification with integer labels, `sparse_categorical_crossentropy` is appropriate.

In [ ]:
model_gap.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

### 4. Train the Model

Now, we train the model using the training data and validate it with the test data. Training will take some time depending on your hardware.

In [ ]:
history = model_gap.fit(train_images, train_labels, epochs=10, # You can increase epochs for better accuracy
                    validation_data=(test_images, test_labels),
                    batch_size=64)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 480s 605ms/step - accuracy: 0.5410 - loss: 1.2825 - val_accuracy: 0.5677 - val_loss: 1.2610
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 452s 578ms/step - accuracy: 0.6938 - loss: 0.8753 - val_accuracy: 0.6867 - val_loss: 0.9425
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 500s 576ms/step - accuracy: 0.7493 - loss: 0.7159 - val_accuracy: 0.6890 - val_loss: 0.9114
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 495s 568ms/step - accuracy: 0.7861 - loss: 0.6187 - val_accuracy: 0.7182 - val_loss: 0.8585
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 507s 574ms/step - accuracy: 0.8075 - loss: 0.5541 - val_accuracy: 0.8035 - val_loss: 0.5754
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 501s 573ms/step - accuracy: 0.8238 - loss: 0.5108 - val_accuracy: 0.8102 - val_loss: 0.5686
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 446s 570ms/step - accuracy: 0.8366 - loss: 0.4684 - val_accuracy: 0.8147 - val_loss: 0.5570
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 453s 580ms/step - accuracy: 0.8451 -

### 5. Evaluate the Model

Finally, we evaluate the model's performance on the test set and visualize the training history.

In [ ]:
test_loss, test_acc = model_gap.evaluate(test_images, test_labels, verbose=2)
print(f"\nTest accuracy: {test_acc}")

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')

plt.show()